# Figure 5 – Correlated Dephasing

Non-Markovian dephasing characterization on **ibm_algiers** (27 qubits).

Panels: FTTPS / PSD (a,b) | DD decay α=2 / α=0 (c,d) | qubit-7 T2 fit (e) | A histogram (f).

Each panel is saved as a separate PDF for assembly in the paper.

In [1]:
import numpy as np
import pickle as pk
import scipy.optimize as spopt
from matplotlib import pyplot as plt
import matplotlib as mpl

%matplotlib inline
mpl.rc('text', usetex=True)
mpl.rc('font', family='serif')

from imports_IBM_NM import (plot_colors, noise_characterization,
                             gen_T2DD_circs, cirq2qiskit)

colors_blais = ['#33658A', '#86BBD8']

## Section 1 – Circuit Generation

T2 Ramsey (`n_pulses=0`), Hahn-Echo (`n_pulses=1`), and dynamical-decoupling circuits (2, 4, 6 pulses) are generated with `gen_T2DD_circs` and converted to Qiskit via `cirq2qiskit`.

In [ ]:
# noise_characterization object for ibm_algiers, qubit 8 (same as fig_02)
# Constructor only sets T1_time/T2_time from a live backend;
# set them manually here as plain attributes.
nc = noise_characterization(m_FTTPS=6, num_T1=30, num_T2=50)
nc.T1_time = 116   # us
nc.T2_time = 106   # us
nc.num_FPW = 50

# Generate circuits for T2*, Hahn-Echo, and DD sequences
n_pulses_list = [0, 1, 2, 4, 6]   # 0 = Ramsey, 1 = HE, 2/4/6 = DD
circ_labels   = ['T2*', 'HE', 'DD-2', 'DD-4', 'DD-6']

dd_circs = {}
for n in n_pulses_list:
    dd_circs[n] = [cirq2qiskit(c) for c in gen_T2DD_circs(n, nc)]

print(f'Generated {len(dd_circs[0])} circuits per sequence.')
print('Example Hahn-Echo circuit:')
print(dd_circs[1][10])

## Section 2 – Load Experiment Data

Characterization data from `ps_exps_algiers_char.p`: T2 Ramsey and Hahn-Echo decay for all 27 qubits on ibm_algiers.

In [ ]:
ps_exps_all = pk.load(open('../data/ps_exps_algiers_char.p', 'rb'))
exp_type    = list(ps_exps_all[0].keys())
x_vals_char = {exp: np.arange(len(ps)) for exp, ps in ps_exps_all[0].items()}

# Time axis for T2 experiments (integer step index; 1 step ~ 2 us)
tt     = x_vals_char['t2']            # shape (50,)
tt_he  = x_vals_char['t2he']          # shape (50,)
tt_fine = np.linspace(tt[0], tt[-1], len(tt)*10)

print(f'Experiment types : {exp_type}')
print(f'Qubits available : {sorted(ps_exps_all.keys())}')
print(f'T2 time axis     : {len(tt)} points, steps 0..{tt[-1]}')

## Section 3 – Fit T2 and Hahn-Echo Decays

Stretched-exponential models:
- **Hahn-Echo**: $p(\tau)=\tfrac{1}{2}\!\left[1+(1-2s)\,e^{-A(\tau/\tau_{\max})^\alpha}\right]$
- **T2 Ramsey**: $p(\tau)=\tfrac{1}{2}\!\left[1+(1-2s)\,e^{-A(\tau/\tau_{\max})^\alpha}\cos(c\tau)\cos(d\tau)\right]$

SPAM offset $s$ is estimated from the first data point. $A$ and $\alpha$ are the main fit parameters; $c$, $d$ capture TLS oscillations in the Ramsey signal.

In [ ]:
def fit_fun_t2he(t, s, A, alpha):
    return (1 + (1 - 2*s) * np.exp(-A * (t / tt[-1])**alpha)) / 2

def fit_fun_t2(t, s, A, alpha, c, d):
    return (1 + (1 - 2*s) * np.exp(-A * (t / tt[-1])**alpha) * np.cos(c*t) * np.cos(d*t)) / 2

# ── Fit qubit 7 (shown in panel e) ──────────────────────────────────────
qb = 7
s  = 1 - (ps_exps_all[qb]['t2he'][0] + ps_exps_all[qb]['t2'][0]) / 2
print(f'Qubit {qb}: SPAM offset s = {s:.4f}')

# Hahn-Echo: free params = (A, alpha)
popt_he, _ = spopt.curve_fit(
    lambda t, A, alpha: fit_fun_t2he(t, s, A, alpha),
    tt, ps_exps_all[qb]['t2he'],
    p0=[1.0, 0.8], bounds=([0, 0.1], [20, 2]))
A_he, alpha_he = popt_he
print(f'HE  fit: A={A_he:.3f}, alpha={alpha_he:.3f}')

# T2 Ramsey: free params = (A, alpha, c, d)
popt_t2, _ = spopt.curve_fit(
    lambda t, A, alpha, c, d: fit_fun_t2(t, s, A, alpha, c, d),
    tt, ps_exps_all[qb]['t2'],
    p0=[5.0, 0.8, 1.5, 0.6], bounds=([0, 0.1, 0, 0], [50, 2, 10, 10]))
A_t2, alpha_t2, b1, b2 = popt_t2
print(f'T2  fit: A={A_t2:.3f}, alpha={alpha_t2:.3f}, c={b1:.3f}, d={b2:.3f}')

## Section 4 – Decay Exponent $A$ for All Qubits

Fit `fit_fun_t2he` and `fit_fun_t2` for each qubit to build the histogram in panel (f).

Note: several qubits have strongly oscillating T2 signals with multiple local minima; the pre-stored reference values from `data_FIG5.p` are used where the free fit diverges.

In [ ]:
# Reference fit values (pre-computed for all 15 analysed qubits)
As_he_ref, As_t2_ref = pk.load(open('../data/data_FIG5.p', 'rb'))
qubit_list = sorted(As_he_ref.keys())

As_he = {}   # Hahn-Echo stretched-exp amplitude
As_t2 = {}   # T2 Ramsey stretched-exp amplitude

print(f"{'Qb':>4}  {'A_he_fit':>10}  {'A_he_ref':>10}  {'A_t2_fit':>10}  {'A_t2_ref':>10}")
print('-' * 54)
for qb in qubit_list:
    if qb not in ps_exps_all:
        As_he[qb] = float(As_he_ref[qb])
        As_t2[qb] = float(As_t2_ref[qb])
        continue
    s_qb = 1 - (ps_exps_all[qb]['t2he'][0] + ps_exps_all[qb]['t2'][0]) / 2
    try:
        phe, _ = spopt.curve_fit(
            lambda t, A, al: fit_fun_t2he(t, s_qb, A, al),
            tt, ps_exps_all[qb]['t2he'], p0=[1.0, 0.8],
            bounds=([0, 0.1], [20, 2]))
        pt2, _ = spopt.curve_fit(
            lambda t, A, al, c, d: fit_fun_t2(t, s_qb, A, al, c, d),
            tt, ps_exps_all[qb]['t2'], p0=[5.0, 0.8, 1.5, 0.6],
            bounds=([0, 0.1, 0, 0], [50, 2, 10, 10]))
        As_he[qb] = phe[0]
        As_t2[qb] = pt2[0]
    except Exception:
        As_he[qb] = float(As_he_ref[qb])
        As_t2[qb] = float(As_t2_ref[qb])
    print(f"{int(qb):>4}  {As_he[qb]:>10.3f}  {float(As_he_ref[qb]):>10.3f}  "
          f"{As_t2[qb]:>10.3f}  {float(As_t2_ref[qb]):>10.3f}")

## Section 5 – Load Pre-computed Simulation Data

Panels (a)–(d) require multi-step numerical results (FTTPS filter integrals, PSD estimates, DD trajectory simulations) that are stored as pre-computed pickles.

In [ ]:
# Panel (a): FTTPS overlap integrals
ww_fttps, PSD_fttps, PSD_fttps_fit, ps_fttps, p_k = pk.load(
    open('../data/figdata-corr_deph-FTTPS_PSD.p', 'rb'))
num_FTTPS = len(ps_fttps)
print(f'FTTPS: {num_FTTPS} sequences, p_k: {len(p_k)} points')

# Panel (b): Power spectral densities
ww, PSD_a2, PSD_a2_fit, PSD_wn = pk.load(
    open('../data/figdata-corr_deph-PSDs.p', 'rb'))
print(f'PSD  : {len(ww)} frequency points')

# Panels (c,d): DD decay simulations vs experiment
exp_type_dd, x_vals_dd, x_vals2, p_k_dd, p_k_markov, ps_1f0, ps_dd = pk.load(
    open('../data/figdata-corr_deph-DD.p', 'rb'))
markers = ['.', 'o', 'x', '^']
print(f'DD   : exp types = {exp_type_dd[1:5]}')
print(f'       p_k_dd keys = {list(p_k_dd.keys())}')
print(f'       p_k_markov keys = {list(p_k_markov.keys())}')

## Section 6 – Paper Figure

Combined 3×2 figure matching the original layout.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(8, 8), dpi=300)

# ── Panel (a): FTTPS overlap p_k ──────────────────────
ax = axes[0, 0]
ax.plot(ps_fttps[:num_FTTPS//2], marker='o', mfc='None',
        color=plot_colors[0], ls='')
ax.plot(p_k, color=plot_colors[0], ls='-')
ax.set_ylabel(r'$p_k(\tau)$', size=20)
ax.set_xlabel('$k$', size=20)
ax.tick_params(labelsize=14)
ax.text(0, .94, '(a)', size=20)

# ── Panel (b): PSDs with inset ─────────────────────────────
ax = axes[0, 1]
dt_p = 0.035
dw   = 0.7
ax.plot(ww, PSD_a2_fit, color='orchid', ls='-', lw=2,
        label=r'$q_0\,(\alpha=2)$')
ax.plot(ww, PSD_wn,     color='slateblue', ls='-', lw=2,
        label=r'$q_2\,(\alpha=0)$')
ax.axvline(dw*dt_p/np.pi, c='k', lw=1, ls='--')
ax.text(9e-3, 9.5e-3, r'$\delta\omega$', size=16)
ax.set_ylabel(r'$S(\omega)$ (MHz)', fontsize=20)
ax.set_xlabel(r'$\omega\,\delta t$', fontsize=20)
ax.set_xscale('log')
ax.yaxis.offsetText.set_fontsize(16)
ax.set_ylim(-.5e-3, 1.1e-2)
ax.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))
ax.tick_params(labelsize=14)
ax.text(4e-4, 9.2e-3, '(b)', size=20)
ax.legend(frameon=False, fontsize=14)

# ── Panel (c): DD decay α=2 ───────────────────────────────────
ax = axes[1, 0]
colors_c = ['blue', 'mediumslateblue', 'slateblue', 'darkslateblue']
markers  = ['.', 'o', 'x', '^']
for i, (pulses, ps) in enumerate(p_k_markov.items()):
    tt_sim = np.linspace(0, 50, 14) if i > 0 else x_vals2
    ax.plot(tt_sim, ps, color=colors_c[i], lw=1.5)
for i, exp in enumerate(exp_type_dd[1:5]):
    ax.plot(x_vals2, ps_1f0[exp], color=colors_c[i],
            lw=0, marker=markers[i], alpha=1, label=2*i)
ax.text(0, 1.05, '(c)', size=20)
ax.set_ylabel(r'$p(\tau)$', fontsize=20)
ax.set_xlabel(r'$\tau\,(\mu s)$', fontsize=20)
ax.set_ylim(0.45, 1.15)
ax.set_xticks([0, 25, 50])
ax.set_yticks([0.6, 0.8, 1.0])
ax.tick_params(labelsize=16)
ax.legend(frameon=False, fontsize=14,
          title=r'Pulses ($\alpha=2$)', title_fontsize=14, ncol=2, loc=0)

# ── Panel (d): DD decay α=0 ───────────────────────────────────
ax = axes[1, 1]
colors_d = ['violet', 'darkviolet', 'orchid', 'purple']
exp_ref  = exp_type_dd[1]
for i, (pulses, ps) in enumerate(p_k_dd.items()):
    ax.plot(x_vals_dd[exp_ref], ps, color=colors_d[i], lw=1.5)
for i, exp in enumerate(exp_type_dd[1:5]):
    ax.plot(x_vals_dd[exp], ps_dd[exp], color=colors_d[i],
            lw=0, marker=markers[i], alpha=1, label=2*i)
ax.text(0, 1.05, '(d)', size=20)
ax.set_ylabel(r'$p(\tau)$', fontsize=20)
ax.set_xlabel(r'$\tau\,(\mu s)$', fontsize=20)
ax.set_ylim(0.45, 1.15)
ax.set_xticks([0, 25, 50])
ax.set_yticks([0.6, 0.8, 1.0])
ax.tick_params(labelsize=14)
ax.legend(frameon=False, fontsize=14,
          title=r'Pulses ($\alpha=0$)', title_fontsize=14, ncol=2, loc=0)

# ── Panel (e): qubit 7 T2HE + T2 with fit ───────────────────────────
ax = axes[2, 0]
s  = 1 - (ps_exps_all[qb]['t2he'][0] + ps_exps_all[qb]['t2'][0]) / 2
ax.plot(tt, ps_exps_all[qb]['t2he'], color=plot_colors[2], lw=2,
        ls='', marker='o', mfc='none', alpha=1, label='Hahn-Echo')
ax.plot(tt_fine, fit_fun_t2he(tt_fine, s, A_he, alpha_he), lw=2, color=plot_colors[2])
ax.plot(tt, ps_exps_all[qb]['t2'], color=plot_colors[1], lw=2,
        ls='', marker='o', mfc='none', alpha=1, label='Ramsey')
ax.plot(tt_fine, fit_fun_t2(tt_fine, s, A_t2, alpha_t2, b1, b2), lw=2, color=plot_colors[1])
ax.set_ylabel(r'$p(\tau)$', fontsize=20)
ax.set_xlabel(r'$\tau\,(\mu s)$', fontsize=20)
ax.set_ylim(0.2, 1.2)
ax.legend(frameon=False, fontsize=14)
ax.tick_params(labelsize=14)
ax.text(-0.3, 1.05, '(e)', size=20)

# ── Panel (f): histogram of A over all qubits ────────────────────────
ax = axes[2, 1]
ax.hist(list(As_he.values()), alpha=0.6, bins=12,
        edgecolor=plot_colors[2], color=plot_colors[2], label='Hahn-Echo')
ax.hist(list(As_t2.values()), alpha=0.6, bins=15,
        edgecolor=plot_colors[1], color=plot_colors[1], label='Ramsey', zorder=-1)
ax.set_ylabel('Counts', size=20)
ax.set_xlabel(r'Decay Power $A$', size=20)
ax.tick_params(labelsize=14)
ax.legend(frameon=False, fontsize=14)
ax.set_xlim(left=-3)
ax.text(-2, 7, '(f)', size=20)

plt.tight_layout()

# ── Inset on panel (b) ────────────────────────────────────────────────
left, bottom, width, height = [0.26, 0.8, 0.22, 0.12]
ax_inset = fig.add_axes([left, bottom, width, height])
ax_inset.plot(ww_fttps, PSD_fttps_fit, color=colors_blais[0], ls='-', lw=2,
              label=r'$\alpha=2$')
ax_inset.set_xlabel(r'$\omega\,\delta t$', size=16)
ax_inset.set_ylabel(r'$S_c(\omega)$', size=16)
ax_inset.set_xscale('log')
ax_inset.set_xlim(left=2e-3)

plt.savefig('../figures/fig_05_correlated_dephasing.pdf', bbox_inches='tight')
plt.show()
print('Saved: ../figures/fig_05_correlated_dephasing.pdf')